# SDS 01 - Hashing & LSH Reference
PySpark + Python standard library only. Covers deterministic sampling, Bloom sizing, MinHash, cosine hyperplane LSH and parameter verification.

In [ ]:
import math, random, hashlib
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.ml import Pipeline
from pyspark.ml.feature import RegexTokenizer, StopWordsRemover, HashingTF, IDF, Normalizer

## 1. Deterministic p% entity sample

In [ ]:
def sample_entities(df,id_col='tracker_id',percent=1):
    return df.filter(F.pmod(F.xxhash64(F.col(id_col)),F.lit(100)) < F.lit(int(percent)))

## 2. Universal hash toy

In [ ]:
def universal_hash(x,a,b,p,m):
    return ((a*x+b)%p)%m

## 3. Bloom-filter parameters

In [ ]:
def bloom_parameters(n,false_positive_rate):
    m=math.ceil(-n*math.log(false_positive_rate)/(math.log(2)**2))
    k_real=(m/n)*math.log(2)
    k=max(1,round(k_real))
    actual=(1-math.exp(-k*n/m))**k
    return {'m_bits':m,'k_hashes':k,'k_real':k_real,'approx_fpr':actual}
print(bloom_parameters(500,0.02))

## 4. Simple Bloom filter using standard-library SHA-256
Reference implementation only; for DataFrame hashing prefer Spark `xxhash64` with salts.

In [ ]:
class BloomFilter:
    def __init__(self,n,fpr):
        p=bloom_parameters(n,fpr); self.m=p['m_bits']; self.k=p['k_hashes']; self.bits=bytearray((self.m+7)//8)
    def _pos(self,value):
        raw=str(value).encode()
        for seed in range(self.k):
            d=hashlib.sha256(seed.to_bytes(4,'little')+raw).digest()
            yield int.from_bytes(d[:8],'little')%self.m
    def add(self,value):
        for pos in self._pos(value): self.bits[pos//8] |= 1<<(pos%8)
    def might_contain(self,value):
        return all((self.bits[pos//8]>>(pos%8))&1 for pos in self._pos(value))

## 5. Shingles and Jaccard

In [ ]:
def shingles(text,k=3):
    text=' '.join(text.lower().split())
    return {text[i:i+k] for i in range(max(0,len(text)-k+1))}

def jaccard(a,b):
    a,b=set(a),set(b)
    return 1.0 if not a and not b else len(a&b)/len(a|b)

## 6. MinHash signature

In [ ]:
def stable_int(token):
    return int.from_bytes(hashlib.sha256(str(token).encode()).digest()[:8],'little')

def minhash_signature(items,num_hashes=100,prime=2305843009213693951,seed=42):
    rng=random.Random(seed); pars=[(rng.randrange(1,prime),rng.randrange(prime)) for _ in range(num_hashes)]
    xs=[stable_int(x)%prime for x in items]
    return [min((a*x+b)%prime for x in xs) for a,b in pars] if xs else [None]*num_hashes

def signature_similarity(a,b):
    pairs=[(x,y) for x,y in zip(a,b) if x is not None and y is not None]
    return sum(x==y for x,y in pairs)/len(pairs) if pairs else 1.0

## 7. LSH amplification

In [ ]:
def amplify(base_p,r,b):
    return 1-(1-base_p**r)**b

def minhash_candidate_probability(jaccard_s,r,b):
    return amplify(jaccard_s,r,b)

## 8. Cosine random-hyperplane LSH

In [ ]:
def hyperplane_base_collision(cosine_s):
    return 1-math.acos(cosine_s)/math.pi

def cosine_candidate_probability(cosine_s,r,b):
    return amplify(hyperplane_base_collision(cosine_s),r,b)

for s in [0.4,0.6,0.8]: print(s,hyperplane_base_collision(s))

## 9. Integer parameter solver / verifier
Directly reusable when the problem specifies high/low similarity and collision targets.

In [ ]:
def choose_cosine_lsh_params(s_hi,target_hi,s_lo,target_lo,r_max=80):
    p_hi=hyperplane_base_collision(s_hi); p_lo=hyperplane_base_collision(s_lo); feasible=[]
    for r in range(1,r_max+1):
        x=p_hi**r
        if x<=0: continue
        b=math.ceil(math.log(1-target_hi)/math.log1p(-x))
        hi=amplify(p_hi,r,b); lo=amplify(p_lo,r,b)
        if hi>=target_hi and lo<=target_lo:
            feasible.append({'bits':b*r,'r':r,'b':b,'P_hi':hi,'P_lo':lo})
    return min(feasible,key=lambda z:z['bits']) if feasible else None

def verify(r,b,s_hi=.60,target_hi=.80,s_lo=.40,target_lo=.05):
    hi=cosine_candidate_probability(s_hi,r,b); lo=cosine_candidate_probability(s_lo,r,b)
    return {'bits':r*b,'P_hi':hi,'hi_ok':hi>=target_hi,'P_lo':lo,'lo_ok':lo<=target_lo}

print('exam LLM pair',verify(31,81700))
print('feasible',choose_cosine_lsh_params(.60,.80,.40,.05))

## 10. Local random-hyperplane signature (for understanding / small demos)

In [ ]:
def make_hyperplanes(dim,n_planes,seed=42):
    rng=random.Random(seed)
    return [[rng.gauss(0,1) for _ in range(dim)] for _ in range(n_planes)]

def hyperplane_signature(vector,planes):
    return tuple(1 if sum(x*w for x,w in zip(vector,p))>=0 else 0 for p in planes)

def band_keys(signature,r):
    assert len(signature)%r==0
    return [(i//r,signature[i:i+r]) for i in range(0,len(signature),r)]

## 11. Spark TF-IDF model: fit on corpus only

In [ ]:
def fit_tfidf(corpus,text_col='text',num_features=1<<16):
    pipe=Pipeline(stages=[RegexTokenizer(inputCol=text_col,outputCol='tokens',pattern=r'\W+'),
        StopWordsRemover(inputCol='tokens',outputCol='filtered'),
        HashingTF(inputCol='filtered',outputCol='tf',numFeatures=num_features),
        IDF(inputCol='tf',outputCol='tfidf'),Normalizer(inputCol='tfidf',outputCol='features',p=2.0)])
    model=pipe.fit(corpus)
    return model,model.transform(corpus)

## 12. Exact cosine fallback for a small corpus
Collect only the ONE query vector, broadcast it, and score corpus vectors in Spark. This is exact search, not LSH.

In [ ]:
def exact_cosine_topk(model,corpus_features,query_text,id_col='id',text_col='text',k=10):
    qdf=spark.createDataFrame([('QUERY',query_text)],[id_col,text_col])
    qvec=model.transform(qdf).select('features').first()['features']
    bq=spark.sparkContext.broadcast(qvec)
    @F.udf(T.DoubleType())
    def dot_with_query(v):
        return float(v.dot(bq.value))
    return (corpus_features.withColumn('cosine',dot_with_query('features'))
            .orderBy(F.desc('cosine')).select(id_col,'cosine').limit(k))

## 13. Actual-exam corpus hygiene reminder
The Gutenberg tree contained `/old/` files and multiple text variants per book. Clean/deduplicate to one canonical book before fitting IDF. Do not fit the query together with the corpus.

## 14. Critique checklist
Correct LSH family? Correct base collision formula? Explicit parameters? Rounded parameters meet constraints? Signature practical? Code matches prose? Spark-only restriction obeyed? Large data stays distributed? Exact fallback clearly labeled?